In [1]:
import re, math, time
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

In [2]:
report = """# Annual Report

## Table of Contents
1. Medical Research
2. Software Engineering
3. Finance

## Medical Research
The medical team studied the XDR-47 strain across three hospitals. Trials were promising, and a follow-up study begins next year.

## Software Engineering
The software engineering department fixed 247 bugs last year. It shipped four major releases, cut code review time by 30% and raised test coverage to 85%.

## Finance
Revenue grew 12% year over year. Operating costs stayed flat, and the company ended the year with a healthy cash reserve.
"""

with open("report.md", "w", encoding="utf-8") as f:
    f.write(report)

In [3]:
def chunk_by_section(document_text):
    return re.split(r"\n## ", document_text)

def generate_embedding(text, model="voyage-3-large", input_type="query"):
    is_list = isinstance(text, list)
    texts = text if is_list else [text]
    result = client.embed(texts, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.items = []

    def add_vector(self, vector, item):
        self.vectors.append(vector)
        self.items.append(item)

    def search(self, query, k=2):
        def distance(a, b):  # cosine distance: smaller = more similar
            dot = sum(x * y for x, y in zip(a, b))
            na = math.sqrt(sum(x * x for x in a))
            nb = math.sqrt(sum(y * y for y in b))
            return 1 - dot / (na * nb)
        scored = [(self.items[i], distance(query, v)) for i, v in enumerate(self.vectors)]
        return sorted(scored, key=lambda p: p[1])[:k]

In [4]:
with open("report.md", "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(len(chunks), "chunks")

embeddings = generate_embedding(chunks, input_type="document")   # request 1

store = VectorIndex()
for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

5 chunks


In [5]:
user_embedding = generate_embedding("What did the software engineering dept do last year?")  # request 2

results = store.search(user_embedding, 2)
for doc, distance in results:
    print(round(distance, 3), "\n", doc["content"][0:200], "\n")

0.331 
 Software Engineering
The software engineering department fixed 247 bugs last year. It shipped four major releases, cut code review time by 30% and raised test coverage to 85%.
 

0.601 
 Finance
Revenue grew 12% year over year. Operating costs stayed flat, and the company ended the year with a healthy cash reserve.
 

